# Experiment 3b: Zero-shot Priority Classification

Classifies each ticket into one of the 4 `Ticket Priority` levels (Critical / High / Medium / Low) using `facebook/bart-large-mnli` without any fine-tuning.

## Hypothesis:

H1 - Customer feedback reflects, to some extent, the urgency of the issue type. The linguistic content of customer support ticket descriptions encodes sufficient signals to distinguish business priority levels - Critical, High, Medium, Low

## Pipeline: 

**load data + 3a predictions → BART-MNLI zero-shot → assemble final CSV.**

**Depends on:** `Experiment_3a_TopicClassification.ipynb` must be run first.

In [1]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
PROJECT_ROOT =  Path.cwd().parents[2]

INPUT_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR   = PROJECT_ROOT / 'results' / 'Experiment_3_BERTMNLI'
SUBJECT_PREDS = RESULTS_DIR / 'Experiment_3a_subject_predictions.csv'
OUT_PATH      = RESULTS_DIR / 'Experiment_3_predictions.csv'

PRIORITY_LABELS = ['Critical', 'High', 'Medium', 'Low']

In [3]:
df = pd.read_csv(INPUT_PATH)
subject = pd.read_csv(SUBJECT_PREDS)

In [4]:
DEVICE = 0 

classifier = pipeline(
    task='zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE
)

Device set to use cuda:0


In [ ]:
ds = Dataset.from_dict({'text': subject['text'].tolist()})

pred_priorities = []
for result in classifier(
    KeyDataset(ds, 'text'),
    candidate_labels=PRIORITY_LABELS,
    truncation=True,
    max_length=256,
    batch_size=1
):
    pred_priorities.append(result['labels'][0])


In [ ]:
out = pd.DataFrame({
    'text': subject['text'].values,
    'pred_subject': subject['pred_subject'].values,
    'pred_subject_conf': subject['pred_subject_conf'].values,
    'pred_priority':pred_priorities,
    'true_subject': subject['true_subject'].values,
    'true_priority': df['Ticket Priority'].values
})
out.to_csv(OUT_PATH, index=False, encoding='utf-8')

out[['pred_subject', 'pred_subject_conf', 'pred_priority', 'true_subject', 'true_priority']].head()

,pred_subject,pred_subject_conf,pred_priority,true_subject,true_priority
0,Product setup,0.8699,High,Product setup,Critical
1,Peripheral compatibility,0.7929,Critical,Peripheral compatibility,Critical
2,Network problem,0.8947,Critical,Network problem,Low
3,Account access,0.7310,Critical,Account access,Low
4,Data loss,0.5145,Low,Data loss,Low


In [ ]:
accuracy = accuracy_score(out['true_priority'], out['pred_priority'])
f1_macro = f1_score(out['true_priority'], out['pred_priority'], average='macro', zero_division=0)
f1_weighted = f1_score(out['true_priority'], out['pred_priority'], average='weighted', zero_division=0)

print(f'Accuracy: {accuracy:.3f}')
print(f'F1 Score: {f1_macro:.3f}')

Accuracy: 0.252
F1 Score: 0.182


## Conclusion & Exploration:  

The zero-shot classification results refute our hypothesis. 

BART-MNLI achieved an accuracy of 25.2%(close to uniform four-class distribution)and a macro F1 of 0.182 across four priority levels, which is statistically equivalent to random chance.

Results indicate that priority labels in this dataset are not inferable from ticket text semantics, consistent with the hypothesis that priority assignment reflects internal operational criteria rather than customer-expressed urgency. 

However, in real-world business scenarios, the semantic information within the text often influences the company's priority in addressing the issue. For example, issues like platform crashes and customer feedback conveying urgency are typically marked as high-priority.

Therefore, we attempt to apply this method to the larger dataset provided in the problem statement.

## Zero-shot Priority Classification — Larger Dataset

In [ ]:
LARGE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'large_dataset_preprocessed.csv'
LARGE_PRIORITY_LABELS = ['critical', 'high', 'medium', 'low', 'very_low']

df_large = pd.read_csv(LARGE_PATH)

Large dataset: 28243 rows
priority
medium    11557
high      10912
low        5774
Name: count, dtype: int64


In [26]:
# Normalise whitespace to avoid BART EOS token mismatch error in batched inference
texts_large = (
    df_large['text_cleaned']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

ds_large = Dataset.from_dict({'text': texts_large.tolist()})

pred_large = []
for result in classifier(
    KeyDataset(ds_large, 'text'),
    candidate_labels=LARGE_PRIORITY_LABELS,
    truncation=True,
    max_length=128,
    batch_size=32
):
    pred_large.append(result['labels'][0])

In [27]:
acc_l     = accuracy_score(df_large['priority'], pred_large)
f1_macro_l = f1_score(df_large['priority'], pred_large, average='macro',    zero_division=0)
f1_wtd_l   = f1_score(df_large['priority'], pred_large, average='weighted', zero_division=0)

print(f'Accuracy: {acc_l:.3f}')
print(f'F1 macro: {f1_macro_l:.3f}')
print(f'F1 weighted: {f1_wtd_l:.3f}')

Accuracy: 0.388
F1 macro: 0.229
F1 weighted: 0.254


In [28]:
pd.DataFrame({
    'true': df_large['priority'].values,
    'pred': pred_large
}).to_csv(RESULTS_DIR / 'Experiment_3_zeroshot_large_predictions.csv', index=False)

Saved.
